# Генерация картинок: `zombie_ep01`

**Что делать (один раз):**
1. **Colab:** меню *Среда выполнения → Сменить среду → T4 GPU*. **Kaggle:** справа *Settings → Accelerator → GPU T4*, и *Internet → On*.
2. Меню *Выполнить всё* (Colab: *Среда выполнения → Выполнить все*; Kaggle: *Run All*).
3. Ждать. Всего картинок: **90**, примерно 25–35 с на картинку на T4 → **~45 мин**.
4. В конце скачается (Colab) или появится в *Output* (Kaggle) архив `zombie_ep01_images.zip`. Пришли его мне.

Если сессия оборвалась — просто запусти всё заново: готовые картинки не перегенерируются (в Colab они сохраняются на Google Диск, если разрешишь доступ в ячейке 2).

In [ ]:
# 1. Установка библиотек (2–3 минуты)
!pip -q install -U diffusers transformers accelerate safetensors "rembg[gpu]" compel pillow
import torch
assert torch.cuda.is_available(), "Нет GPU! Включи T4 в настройках среды и перезапусти."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Папка для результатов (Colab: можно сохранить на Google Диск, чтобы не терять при обрыве)
import os
SLUG = 'zombie_ep01'
OUT = None
if os.path.exists("/kaggle/working"):
    OUT = f"/kaggle/working/{SLUG}"
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = f"/content/drive/MyDrive/{SLUG}"
    except Exception as e:
        print("Диск не подключён, сохраняю локально:", e)
        OUT = f"/content/{SLUG}"
os.makedirs(OUT, exist_ok=True)
print("Сохраняю в", OUT)

In [ ]:
# 3. Список картинок (собран режиссёром; не редактируй без нужды)
ASSETS = {
 "style_prefix": "moody 2D digital painting, serious survival drama, soft painted shading, heavily desaturated cold blue-grey palette, deep shadows, thin pale rim light, realistic adult proportions, muted tired faces, worn grimy surfaces, clean readable shapes",
 "negative": "text, letters, words, numbers, logo, watermark, signature, caption, photo, photorealistic, 3d render, anime, chibi, cartoon comedy, bright colors, saturated colors, neon, lens flare, bokeh, glossy, gore, blood, wounds, exposed flesh, corpse, glowing eyes, red eyes, open mouth, teeth, smiling, big eyes, extra fingers, extra limbs, deformed hands, fused fingers, asymmetrical face, cropped head, cropped feet, duplicate, multiple views, character sheet, split image, border, frame, busy background, full red tint",
 "assets": [
  {
   "id": "bg_window_yard",
   "kind": "bg",
   "prompt": "Inside a ground floor apartment room at night, plain old window with a thin frame in the center of the wall at standing eye height, window sill low, through the glass a dark residential courtyard at the same level, bare trees, a bench, dark panel block across the yard, thin faded curtain pushed aside, bare worn wall, wooden floor in the lower third, vertical composition, empty room, no people, neutral dim light, no light source turned on, flat frontal view, minimal perspective distortion",
   "seed": 70101,
   "variants": 4,
   "note": "Сц. 1, 3, 4, 9 (начало). Окно ДО: открыто взгляду, двор на уровне глаз. Стекло потом маскируем полигоном, чтобы за ним поставить силуэт заражённого."
  },
  {
   "id": "bg_window_sealed",
   "kind": "bg",
   "prompt": "Inside a ground floor apartment room at night, the same plain old window in the center of the wall, now firmly shut, heavy dark wool curtain drawn tight over the whole window, only a faint pale edge of cold light at the curtain border, bare worn wall, wooden floor in the lower third, vertical composition, empty room, no people, neutral dim light, no light source turned on, flat frontal view, minimal perspective distortion",
   "seed": 70101,
   "variants": 4,
   "note": "Сц. 9 (после склейки), 10. Окно ПОСЛЕ: закрыто и зашторено. Тот же seed, что у ДО; лучше img2img/inpaint от выбранного ДО (strength ~0.5), чтобы совпала перспектива."
  },
  {
   "id": "bg_facade_aquarium",
   "kind": "bg",
   "prompt": "Exterior night view from a dark courtyard of a ground floor window of an old panel apartment block, the window at human head height right above the ground, glass reflecting faintly, behind the glass a dim apartment interior visible like a fish tank, wall, doorway, furniture shapes, cold bluish glow inside, dark wet asphalt and bare bush in front, vertical composition, no people, neutral dim light, no light source turned on, flat frontal view, minimal perspective distortion",
   "seed": 70201,
   "variants": 3,
   "note": "Сц. 2 «аквариум»: квартира за стеклом. Силуэты жильцов внутри ставим кодом (тёмные копии спрайтов)."
  },
  {
   "id": "bg_facade_dark",
   "kind": "bg",
   "prompt": "Exterior night view from a dark courtyard of a ground floor window of an old panel apartment block, the window at human head height right above the ground, window completely dark, curtain drawn inside, nothing visible behind the glass, all neighbouring windows dark, dark wet asphalt and bare bush in front, vertical composition, no people, neutral dim light, no light source turned on, flat frontal view, minimal perspective distortion",
   "seed": 70201,
   "variants": 3,
   "note": "Сц. 11 (вставка 2 с): снаружи окна тёмные. ПОСЛЕ к bg_facade_aquarium, тот же seed; лучше img2img от выбранного варианта aquarium."
  },
  {
   "id": "bg_hall_kitchen_open",
   "kind": "bg",
   "prompt": "Narrow apartment hallway at night seen straight on, at the far end an open doorway into a small dark kitchen, inside the kitchen an old window with its casement swung wide open into the night, thin curtain moving, coat hooks on the wall, worn linoleum floor in the lower third, vertical composition, empty room, no people, neutral dim light, no light source turned on, flat frontal view, minimal perspective distortion",
   "seed": 70301,
   "variants": 4,
   "note": "Сц. 5 (Тамара Петровна уходит в проём, видна открытая створка), сц. 6 (размытый фон крупного плана)."
  },
  {
   "id": "bg_kitchen_door_ajar",
   "kind": "bg",
   "prompt": "Close view of a closed-looking old painted wooden kitchen door at the end of a dark apartment hallway, the door left slightly ajar with a narrow dark vertical gap, small gap between door bottom and floor, plain door with no handle visible, worn linoleum floor in the lower third, dark walls, vertical composition, empty room, no people, neutral dim light, no light source turned on, flat frontal view, minimal perspective distortion",
   "seed": 70401,
   "variants": 3,
   "note": "Сц. 7: дверь кухни приоткрыта, под ней полоса света (кодом), по полосе проходит тень (кодом). Ручку рисуем кодом."
  },
  {
   "id": "bg_kitchen_door_closed",
   "kind": "bg",
   "prompt": "Close view of an old painted wooden kitchen door at the end of a dark apartment hallway, door fully shut flush with the frame, small gap between door bottom and floor, plain door with no handle visible, worn linoleum floor in the lower third, dark walls, vertical composition, empty room, no people, neutral dim light, no light source turned on, flat frontal view, minimal perspective distortion",
   "seed": 70401,
   "variants": 3,
   "note": "Сц. 7 (после щелчка), 16: дверь закрыта. Тот же seed, что у ajar; лучше inpaint только полотна двери. Ручка — SVG кодом (в сц. 16 медленно опускается)."
  },
  {
   "id": "bg_table_three_chairs",
   "kind": "bg",
   "prompt": "Small apartment living room at night seen straight on, an old square wooden table with a plain tablecloth, exactly three wooden chairs around it, one side of the table has an empty space where a chair is missing, faint scuff marks on the floor there, an empty cup on the table, bare wall, vertical composition, empty room, no people, neutral dim light, no light source turned on, flat frontal view, minimal perspective distortion",
   "seed": 70501,
   "variants": 4,
   "note": "Сц. 8: «четвёртого стула нет». Генератор путает счёт стульев — выбирать вариант, где ровно три, лишний стул можно закрасить."
  },
  {
   "id": "bg_inner_room",
   "kind": "bg",
   "prompt": "Small windowless inner room of an apartment at night, blank wall with old faded wallpaper, two thin mattresses with grey blankets on the floor along the blank wall, a small table lamp on a stool turned off, a narrow wall shelf on the right at chest height, door on the side, no windows at all, wooden floor in the lower third, vertical composition, empty room, no people, neutral dim light, no light source turned on, flat frontal view, minimal perspective distortion",
   "seed": 70601,
   "variants": 4,
   "note": "Сц. 11, 12 (размытый фон под схемой), 13, 15. Свет лампы — тёплым градиентом кодом. Полка справа — сюда ставим свечу (сц. 13)."
  },
  {
   "id": "bg_bath_tap",
   "kind": "bg",
   "prompt": "Old small bathroom at night seen straight on, chipped enamel bathtub and a worn sink with an old metal tap in the center, grey-green tiled wall, a plastic stool in front of the sink, bare and clean, tiled floor in the lower third, vertical composition, empty bathroom, no people, neutral dim light, no light source turned on, flat frontal view, minimal perspective distortion",
   "seed": 70701,
   "variants": 3,
   "note": "Сц. 14: набираем воду. Струю и капли рисуем кодом, банки/канистры — спрайт prop_water_set перед краном."
  },
  {
   "id": "mark_stand_notebook",
   "kind": "sprite",
   "prompt": "Full body standing pose of a thin gaunt man aged 40, short dark hair with grey at temples, light stubble, hollow cheeks, worn zipped grey jacket, dark grey trousers, scuffed black boots, standing still and tense, three-quarter view turned to the left, holding a closed notebook with a dark green oilcloth cover against his chest with both hands, face clearly readable, eyes open, mouth closed and neutral, isolated on a plain flat light grey background, single subject centered, nothing else in the image, even soft neutral light, no cast shadow",
   "seed": 11001,
   "variants": 4,
   "note": "Марк говорит: сц. 4, 10, 12; тёмным силуэтом — сц. 8. Главная поза, лицо читаемо для рта кодом."
  },
  {
   "id": "mark_reach",
   "kind": "sprite",
   "prompt": "Full body side view pose of a thin gaunt man aged 40, short dark hair with grey at temples, light stubble, hollow cheeks, worn zipped grey jacket, dark grey trousers, scuffed black boots, facing right, stepping forward with both arms extended straight ahead pushing something with flat palms, body leaning, head in profile, mouth closed, isolated on a plain flat light grey background, single subject centered, nothing else in the image, even soft neutral light, no cast shadow",
   "seed": 11001,
   "variants": 4,
   "note": "Действие: закрывает окно (сц. 9), закрывает дверь кухни (сц. 7), забирает свечу (сц. 13). Один спрайт на все «толкает/тянется»."
  },
  {
   "id": "mark_write_sit",
   "kind": "sprite",
   "prompt": "Full body pose of a thin gaunt man aged 40, short dark hair with grey at temples, light stubble, hollow cheeks, worn zipped grey jacket, dark grey trousers, scuffed black boots, sitting on a low wooden stool, three-quarter view turned to the right, an open notebook with a dark green oilcloth cover on his knees, writing in it with a pen, head bowed over the page, face clearly readable, eyes open, mouth closed and neutral, isolated on a plain flat light grey background, single subject centered, nothing else in the image, even soft neutral light, no cast shadow",
   "seed": 11001,
   "variants": 3,
   "note": "Сц. 15: пишет правило, перо скребёт."
  },
  {
   "id": "inna_stand",
   "kind": "sprite",
   "prompt": "Full body standing pose of a woman aged 32, short cropped dark brown hair, pale tired skin, dark circles under eyes, dark olive knitted cardigan over a grey t-shirt, dark trousers, white cloth bandage wrapped around her left palm, standing still and listening, three-quarter view turned to the right, arms close to the body, bandaged left hand held against her chest, face clearly readable, eyes open, mouth closed and neutral, isolated on a plain flat light grey background, single subject centered, nothing else in the image, even soft neutral light, no cast shadow",
   "seed": 22002,
   "variants": 4,
   "note": "Инна говорит: сц. 4, 7, 12; у двери — сц. 15; тёмным силуэтом — сц. 8."
  },
  {
   "id": "inna_push",
   "kind": "sprite",
   "prompt": "Full body side view pose of a woman aged 32, short cropped dark brown hair, pale tired skin, dark circles under eyes, dark olive knitted cardigan over a grey t-shirt, dark trousers, white cloth bandage wrapped around her left palm, facing left, pushing something heavy with both hands and her shoulder, leaning forward, bandaged left hand visible, head in profile, mouth closed, isolated on a plain flat light grey background, single subject centered, nothing else in the image, even soft neutral light, no cast shadow",
   "seed": 22002,
   "variants": 3,
   "note": "Сц. 9: двигает шкаф вдоль стены; произносит реплику в профиль."
  },
  {
   "id": "inna_hand",
   "kind": "sprite",
   "prompt": "Close-up of a single adult woman's left hand and wrist, pale tired skin, white cloth bandage wrapped around the palm, palm facing the viewer, fingers together and straight, raised as if covering a mouth, the cuff of a dark olive knitted cardigan at the wrist, natural proportions, five fingers, isolated on a plain flat light grey background, single subject centered, nothing else in the image, even soft neutral light, no cast shadow",
   "seed": 22002,
   "variants": 4,
   "note": "Сц. 6: крупно ладонь Инны поверх lesha_portrait (на месте рта). Исключение: полное описание Инны не вставлено (иначе SDXL рисует всю фигуру), совпадают бинт, рукав кардигана, кожа и seed. Руки — слабое место SDXL, 4 варианта."
  },
  {
   "id": "lesha_wall",
   "kind": "sprite",
   "prompt": "Full body standing pose of a tall lanky teenage boy aged 17, messy light brown hair, thin face, oversized faded dull burgundy knitted sweater too big for him, sleeves over his wrists, dark jeans, grey sneakers, standing pressed flat with his back against an invisible wall, front view, shoulders raised, arms tight at his sides, head turned slightly to the left, face clearly readable, eyes open, mouth closed and neutral, isolated on a plain flat light grey background, single subject centered, nothing else in the image, even soft neutral light, no cast shadow",
   "seed": 33003,
   "variants": 4,
   "note": "Сц. 9 (прижался к стене сбоку), 10 (общий план); тёмным силуэтом — сц. 8."
  },
  {
   "id": "lesha_portrait",
   "kind": "sprite",
   "prompt": "Head and shoulders portrait of a tall lanky teenage boy aged 17, messy light brown hair, thin face, oversized faded dull burgundy knitted sweater too big for him, sleeves over his wrists, dark jeans, grey sneakers, front view, frightened wide-open eyes, eyebrows slightly raised, face clearly readable, eyes open, mouth closed and neutral, isolated on a plain flat light grey background, single subject centered, nothing else in the image, even soft neutral light, no cast shadow",
   "seed": 33003,
   "variants": 4,
   "note": "Сц. 6 (под ладонью Инны), сц. 10 (реплика «Так мы ничего не увидим»). Рот — кодом."
  },
  {
   "id": "lesha_sit_candle",
   "kind": "sprite",
   "prompt": "Full body pose of a tall lanky teenage boy aged 17, messy light brown hair, thin face, oversized faded dull burgundy knitted sweater too big for him, sleeves over his wrists, dark jeans, grey sneakers, sitting on the floor with knees pulled up, three-quarter view turned to the right, holding an unlit plain white wax candle in both hands in front of his chest, looking up, face clearly readable, eyes open, mouth closed and neutral, isolated on a plain flat light grey background, single subject centered, nothing else in the image, even soft neutral light, no cast shadow",
   "seed": 33003,
   "variants": 3,
   "note": "Сц. 13: «Можно свечу?» Свеча незажжённая; пламя не нужно."
  },
  {
   "id": "lesha_sit_hug",
   "kind": "sprite",
   "prompt": "Full body pose of a tall lanky teenage boy aged 17, messy light brown hair, thin face, oversized faded dull burgundy knitted sweater too big for him, sleeves over his wrists, dark jeans, grey sneakers, sitting on the floor with knees pulled up, three-quarter view turned to the right, arms wrapped around his knees, sleeves over his hands, empty hands, face clearly readable, eyes open, mouth closed and neutral, isolated on a plain flat light grey background, single subject centered, nothing else in the image, even soft neutral light, no cast shadow",
   "seed": 33003,
   "variants": 3,
   "note": "Сц. 13 (после того как Марк забрал свечу), сц. 15 (на матрасе)."
  },
  {
   "id": "tamara_walk_kettle",
   "kind": "sprite",
   "prompt": "Full body pose of an elderly woman aged 68, grey hair in a low bun, wrinkled kind face, faded brown checkered wool blanket wrapped over her shoulders, dark navy house dress, felt slippers, a bunch of old keys on a cord around her neck, walking away slowly, three-quarter back view turned to the left, face partly visible in profile, holding an old white enamel kettle in her right hand, isolated on a plain flat light grey background, single subject centered, nothing else in the image, even soft neutral light, no cast shadow",
   "seed": 44004,
   "variants": 4,
   "note": "Сц. 5: берёт чайник и уходит в проём кухни. Единственный спрайт Тамары Петровны; не говорит, рот не нужен."
  },
  {
   "id": "infected_stand",
   "kind": "sprite",
   "prompt": "Full body silhouette of a motionless adult figure standing very still facing forward, head tilted slightly to one shoulder, arms hanging, ordinary coat visible only as a shape, solid near-black body with a thin pale cold rim light along the edges, no face details, no eyes, isolated on a plain flat light grey background, single subject centered, nothing else in the image, even soft neutral light, no cast shadow",
   "seed": 55005,
   "variants": 4,
   "note": "Сц. 3 (и 1, опционально): неподвижная тень во дворе за стеклом. Та же форма — источник тени, проходящей по полосе света в сц. 7 (кодом: чёрный, размытие, растяжение)."
  },
  {
   "id": "prop_wardrobe",
   "kind": "sprite",
   "prompt": "Old tall dark brown wooden wardrobe with two plain doors, front view, worn varnish, scratched corners, no decorations, isolated on a plain flat light grey background, single subject centered, nothing else in the image, even soft neutral light, no cast shadow",
   "seed": 60101,
   "variants": 3,
   "note": "Сц. 9: шкаф едет вдоль стены (кодом, по оси X)."
  },
  {
   "id": "prop_water_set",
   "kind": "sprite",
   "prompt": "Group of closed water containers standing together, three large clear glass jars with plain metal lids and two clear plastic water canisters with plain caps, filled with clean water, no labels, slightly scratched, front view, isolated on a plain flat light grey background, single subject centered, nothing else in the image, even soft neutral light, no cast shadow",
   "seed": 60201,
   "variants": 3,
   "note": "Сц. 14: закрытые банки и канистры под краном."
  },
  {
   "id": "prop_candle",
   "kind": "sprite",
   "prompt": "Single unlit plain white wax candle standing on a small chipped enamel saucer, black wick, front view, isolated on a plain flat light grey background, single subject centered, nothing else in the image, even soft neutral light, no cast shadow",
   "seed": 60301,
   "variants": 2,
   "note": "Сц. 13: Марк ставит свечу на полку. Не зажжена."
  },
  {
   "id": "prop_notebook_open",
   "kind": "sprite",
   "prompt": "Open old notebook with a dark green oilcloth cover lying flat, seen from above, two blank lined pages, slightly worn corners, a pen lying across it, isolated on a plain flat light grey background, single subject centered, nothing else in the image, even soft neutral light, no cast shadow",
   "seed": 60401,
   "variants": 3,
   "note": "Сц. 15: вставка крупно; строки правила пишутся кодом поверх чистых страниц."
  }
 ]
}
print(len(ASSETS["assets"]), "ассетов")

In [ ]:
# 4. Загрузка модели SDXL (≈7 ГБ, 3–5 минут)
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", vae=vae,
    torch_dtype=torch.float16, variant="fp16", use_safetensors=True).to("cuda")
pipe.set_progress_bar_config(disable=True)
# Длинные промпты (>77 токенов) без обрезки — через compel (MIT)
from compel import CompelForSDXL
cmp = CompelForSDXL(pipe)

In [ ]:
# 5. Генерация (пропускает уже готовые файлы)
import time
from rembg import remove, new_session
cut_session = new_session("isnet-general-use")
SIZES = {"bg": (768, 1344), "sprite": (1024, 1024)}
style, neg = ASSETS["style_prefix"], ASSETS["negative"]
t0, done = time.time(), 0
for a in ASSETS["assets"]:
    w, h = SIZES[a["kind"]]
    for v in range(a.get("variants", 1)):
        path = f"{OUT}/{a['id']}_v{v}.png"
        if not os.path.exists(path):
            g = torch.Generator("cuda").manual_seed(int(a["seed"]) + v * 1000)
            # у спрайтов главное — однотонный фон для вырезки, ставим это в начало
            lead = "isolated single subject on a plain flat light grey background, " if a["kind"] == "sprite" else ""
            c = cmp(f"{lead}{style}, {a['prompt']}", negative_prompt=neg)
            img = pipe(prompt_embeds=c.embeds, pooled_prompt_embeds=c.pooled_embeds,
                       negative_prompt_embeds=c.negative_embeds, negative_pooled_prompt_embeds=c.negative_pooled_embeds,
                       width=w, height=h, num_inference_steps=30, guidance_scale=6.5, generator=g).images[0]
            img.save(path)
        if a["kind"] == "sprite":
            cut = f"{OUT}/{a['id']}_v{v}_cut.png"
            if not os.path.exists(cut):
                from PIL import Image
                remove(Image.open(path), session=cut_session).save(cut)
        done += 1
        print(f"[{done}] {a['id']} v{v}  ({time.time()-t0:.0f} с)")
print("Готово")

In [ ]:
# 6. Превью: все варианты одним листом
from PIL import Image
import glob, math
files = sorted(f for f in glob.glob(f"{OUT}/*.png") if not f.endswith("_cut.png"))
th = 220
thumbs = [Image.open(f).convert("RGB") for f in files]
thumbs = [t.resize((int(t.width * th / t.height), th)) for t in thumbs]
cols = 8; rows = math.ceil(len(thumbs) / cols)
sheet = Image.new("RGB", (cols * 240, rows * (th + 10)), "black")
for i, t in enumerate(thumbs):
    sheet.paste(t, ((i % cols) * 240, (i // cols) * (th + 10)))
sheet.save(f"{OUT}/_preview.jpg"); display(sheet)

In [ ]:
# 7. Архив для отправки
import shutil
zip_path = shutil.make_archive(f"/tmp/{SLUG}_images", "zip", OUT)
if os.path.exists("/kaggle/working"):
    shutil.copy(zip_path, "/kaggle/working/"); print("Скачай из Output →", os.path.basename(zip_path))
else:
    from google.colab import files; files.download(zip_path)